# 020 Encoding DNA Sequence as Binary

In [1]:
import gabi.utils as gu
import numpy as np
import pickle
import sys

## 01 Define Encodings
Create the encoding functions


In [3]:
# generic encoder wrapper for mapping strings to encoding dicts
def encode(seq: str, enc: dict) -> np.ndarray: 
    """Encode (into numpy array uint8) a sequence str with and enc(oding) dict"""
    encoded = None
    try:
        encoded = np.array(
        [c for s in seq.upper() for c in enc[s]], # flat unitary list
        dtype="uint8",
        )
    except KeyError as e:
        print(f"Cannot encode nucleotide {e} with this encoder! {enc}", file=sys.stderr)
        return None
    return {
        'seq': encoded,
        'bits': enc['bits'],
    }


### 1. GC

_1-bit_

$$
A \rightarrow 0 \\
C \rightarrow 1 \\
G \rightarrow 1 \\ 
T \rightarrow 0
$$


In [4]:
gc = {
   'A': '0',
   'C': '1',
   'G': '1', 
   'T': '0',
   'bits': 1,
   'id': 'gc',
} 

encode('ACGTX', gc)
encode('ACGT', gc)

Cannot encode nucleotide 'X' with this encoder! {'A': '0', 'C': '1', 'G': '1', 'T': '0', 'bits': 1, 'id': 'gc'}


{'seq': array([0, 1, 1, 0], dtype=uint8), 'bits': 1}

### 2. GC Maximum Entropy
_2-bit_

Relate by the binary derivative. CG, AT related by same binary entropy {0, 1}, the more complex encodings (G,C) which have the higher entropy encodings, are biologically more important or more abundant at least, in coding regions.

Note, this biases GC content with a higher binary entropy, which is based on the 1st to *n-1*th order binary dervitives of a string.

$$
A \rightarrow 00 \\
C \rightarrow 01 \\
G \rightarrow 10 \\ 
T \rightarrow 11
$$

In [5]:
gcme = {
   'A': '00',
   'C': '01',
   'G': '10', 
   'T': '11',
   'bits': 2,
   'id': 'gcme',
} 

encode('ACGT', gcme)

{'seq': array([0, 0, 0, 1, 1, 0, 1, 1], dtype=uint8), 'bits': 2}

### 3. GC Maximum Magnitude
_3-bit_

The binary derivatives are all 1, GC bias to higer magnitude(2).

We can now allow *N*s, unknown bases, to be encoded as all zeros.

$$
A \rightarrow 001 \\
C \rightarrow 011 \\
G \rightarrow 110 \\ 
T \rightarrow 100 \\
N \rightarrow 000
$$

In [6]:
gcmm = {
   'A': '001',
   'C': '011',
   'G': '110', 
   'T': '100',
   'N': '000',
   'bits': 3,
   'id': 'gcmm',
} 

r = encode('ACGTN', gcmm)

r['seq'].reshape(5,3)

array([[0, 0, 1],
       [0, 1, 1],
       [1, 1, 0],
       [1, 0, 0],
       [0, 0, 0]], dtype=uint8)

### 4. Categorical
_4-bit_

A balanced encoding.

_AKA One-Hot encoding_

$$
A \rightarrow 0001 \\
C \rightarrow 0010 \\
G \rightarrow 0100 \\ 
T \rightarrow 1000 \\
N \rightarrow 0000
$$

In [7]:
categorical = {
   'A': '0001',
   'C': '0010',
   'G': '0100', 
   'T': '1000',
   'N': '0000',
   'bits': 4,
   'id': 'categorical',
} 

encode('ACGTN', categorical)

{'seq': array([0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0],
       dtype=uint8),
 'bits': 4}

## 02 Test
Get a few samples to help develop the encoding functions.

Encode some samples.

In [4]:
data = 'data/020.testdatadump.p'
####sgen = gu.sample_generator(
####    species='mus_musculus',
####    n=100,
####    sample_len=10000,
####    feature_types=['exon', 'cds'],
####    biotype='protein_coding',
####    seed=3,
####)
####test_samples = [s async for s in sgen]
####len(test_samples)
####[np.sum(s['labels'], axis=0) for s in test_samples]

####pickle.dump( test_samples, open( data, "wb" ) )
##### don't save again, use the saved samples from earlier

In [5]:
saved_samples = pickle.load(open(data, "rb"))

####for s in saved_samples:
####    for ftype, labels in s['labels'].items():
####        print(f"{ftype}={np.sum(labels)}", end=' ')
####    print()


In [6]:
import gabi.encodings as ge

print(len(saved_samples))
#print(saved_samples[0])
#encode(saved_samples[0]['seq'], gc)
#encoders = [gc, gcme, gcmm, categorical]
force = False
for s in saved_samples:
    if not s.get('encoded'): s['encoded'] = {} # could make samples default dicts...
        
    for encoder in ge.ALL:
        if not s['encoded'].get(encoder['id']) or force: # add override?
            s['encoded'][encoder['id']] = gu.encode(s['seq'], encoder)
        



#encoded_data = 'data/020_encoded_test_data.p'
#pickle.dump( saved_samples, open( encoded_data, "wb" ) )

saved_samples[0]

100


{'seq_region': '12',
 'start': 73045211,
 'end': 73055210,
 'strand': 1,
 'len': 10000,
 'species': 'mus_musculus',
 'loc': '12:73045211..73055210:1',
 'seq': 'CTACATAAGCAATGTCTATCGGCCAAGAGTATACAGTGAGATCTGTCTCCAAACAACAAAACAGTTGCTGTTAACTTCCATACATGAAGATTTTTTTCTATACTTTGAGAGTAAGTATCTTCCTTGAATAGGCCAGATAACTTAATTTAAGGAAAAGCATAAAAGAGATCTCAAAATAGAAAATATAGAGATTATTTTGAGCTTGAGCAGGCCCCAGGTACTCCTGGGACATTTGCTTATGCACCTGGGCATTTATGAAAACTAGAGGAAAAGGAAAGGGGCGTGTGTTGTGACAGTGCGGTCCTCCCATAGTTTCTCTACCCACACCTGACATGCAAGGGTACTTTGCAAAAGAAATCAGTGAAAATGAAAATCTAAGTCATTTGATGTCTGCCTGACTCAGAAAGCAAAACTGGTCTGGACTCCCCCTTTTACCCAAATTGGAGAGAAGTTATCACTTTGTGTCCACAGGGAGTGGAGTGAATAATACTGTGACAACTCAGACCAGTTCCCTCTGAGTCTGAGACCAAAGGCTGTAAATGAAGCCCAAACCAGATCCCAGTGCAAAGGTGAAAACTGACATGATCCACCTCTGCACCAAACTATTCCAGCAAAAGATACACGAGGTCTATTTTTGTCACAATAATAAAACATTAGCTTGTACCCATAGTCTATAAAACAACCTTCACCCAAACACACTTCTTAGTTCATTTGAACGTCTCTATCACCCCAGGAGGAGAGTACTTATGTGTGGCTACTGTAGCTCAAGATGCCTTCTCTATAGCATCCCTCAATAAACACTCTTCCGATTGATGACAAGAGCACATCACCTGTCACATC